<a href="https://colab.research.google.com/github/siddharth-0309/olist-delivery-dashboard/blob/main/olist_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# customers file ----
customers = pd.read_csv('/content/drive/MyDrive/SQL PROJECT /archive/customers.csv')
# order items ----
order_items = pd.read_csv('/content/drive/MyDrive/SQL PROJECT /archive/order_items_dataset.csv')
#  order payments
order_payments  = pd.read_csv('/content/drive/MyDrive/SQL PROJECT /archive/order_payments_dataset.csv')
# order review
order_review  = pd.read_csv('/content/drive/MyDrive/SQL PROJECT /archive/order_reviews_dataset.csv')
# orders
orders = pd.read_csv('/content/drive/MyDrive/SQL PROJECT /archive/olist_orders_dataset.csv')
# products dataset
products_dataset = pd.read_csv('/content/drive/MyDrive/SQL PROJECT /archive/olist_products_dataset.csv')
#sellers dataset
sellers_dataset= pd.read_csv('/content/drive/MyDrive/SQL PROJECT /archive/olist_sellers_dataset.csv')
# product_category_name_translation
product_category_name_translation = pd.read_csv('/content/drive/MyDrive/SQL PROJECT /archive/product_category_name_translation.csv')

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [5]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [6]:
import sqlite3

conn = sqlite3.connect('olist.db')

customers.to_sql('customers', conn, if_exists='replace', index=False)
order_items.to_sql('order_items', conn, if_exists='replace', index=False)
order_payments.to_sql('order_payments', conn, if_exists='replace', index=False)
order_review.to_sql('order_review', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
products_dataset.to_sql('products', conn, if_exists='replace', index=False)
sellers_dataset.to_sql('sellers', conn, if_exists='replace', index=False)
product_category_name_translation.to_sql('category_translation', conn, if_exists='replace', index=False)

print("All tables loaded into olist.db ✅")

All tables loaded into olist.db ✅


In [7]:
import pandas as pd

query = """
SELECT o.order_id, o.order_status, c.customer_state, p.payment_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_payments p ON o.order_id = p.order_id
LIMIT 10;
"""
result = pd.read_sql_query(query, conn)
result

,order_id,order_status,customer_state,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,SP,2.00
1,e481f51cbdc54678b7cc49136f2d6af7,delivered,SP,18.12
2,e481f51cbdc54678b7cc49136f2d6af7,delivered,SP,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,delivered,BA,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,delivered,GO,179.12
5,949d5b44dbf5de918fe9c16f97b45f8a,delivered,RN,72.20
6,ad21c59c0840e6cb83a9ceb5573f8159,delivered,SP,28.62
7,a4591c265e18cb1dcee52889e2d8acc3,delivered,PR,175.26
8,136cce7faa42fdb2cefd53fdc79a6098,invoiced,RS,65.95
9,6514b8ad8028c9f2cc2374ded245783f,delivered,RJ,75.16


# **Problem 1: Delay-Satisfaction Analysis**

In [8]:
import pandas as pd

q1 = """
SELECT
    CASE
        WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <= 0 THEN 'On-Time'
        WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) BETWEEN 1 AND 3 THEN '1-3 Days Late'
        WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) BETWEEN 4 AND 7 THEN '4-7 Days Late'
        ELSE '7+ Days Late'
    END AS delay_bucket,
    COUNT(*) AS total_orders,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM orders o
JOIN order_review r ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY delay_bucket
ORDER BY avg_review_score DESC;
"""

result1 = pd.read_sql_query(q1, conn)
result1

,delay_bucket,total_orders,avg_review_score
0,On-Time,88658,4.29
1,1-3 Days Late,1360,3.51
2,7+ Days Late,5060,2.41
3,4-7 Days Late,1281,2.17


# **Problem 2: Seller Accountability (Window Function)**

In [9]:
q2 = """
WITH seller_stats AS (
    SELECT
        oi.seller_id,
        COUNT(DISTINCT o.order_id) AS total_orders,
        SUM(CASE
                WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
                THEN 1 ELSE 0
            END) AS late_orders,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    JOIN order_review r ON o.order_id = r.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
    GROUP BY oi.seller_id
    HAVING total_orders >= 10
)
SELECT *,
       ROUND(1.0 - (late_orders * 1.0 / total_orders), 3) AS on_time_rate,
       RANK() OVER (ORDER BY (1.0 - (late_orders * 1.0 / total_orders)) DESC) AS seller_rank
FROM seller_stats
ORDER BY seller_rank
LIMIT 15;
"""
# ORDER BY seller_rank DESC for worst
result2 = pd.read_sql_query(q2, conn)
result2

,seller_id,total_orders,late_orders,avg_review_score,on_time_rate,seller_rank
0,013900e863eace745d3ec7614cab5b1a,23,0,4.73,1.0,1
1,014d9a685fd57276679edd00e07089e5,11,0,3.83,1.0,1
2,01bcc9d254a0143f0ce9791b960b2a47,10,0,4.40,1.0,1
3,02c988090b766852e088c69d7fb3b551,12,0,4.69,1.0,1
4,02ecc2a19303f05e59ce133fd923fff7,21,0,4.54,1.0,1
5,06bb3a2fe5e7b7a845b13e8fb91bd944,17,0,4.61,1.0,1
6,07bf9669d84d1f11be443a9dd938f698,18,0,3.25,1.0,1
7,08084d990eb3f53af056ccbc1730c8a7,11,0,4.45,1.0,1
8,08d0949a9a17c027262db1f3c450c26c,12,0,3.54,1.0,1
9,094ced053e257ae8cae57205592d6712,19,0,4.05,1.0,1


# **Problem 3: Geographic Risk (State-wise)**


In [10]:
q3 = """
SELECT
    c.customer_state,
    COUNT(*) AS total_orders,
    SUM(CASE
            WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
            THEN 1 ELSE 0
        END) AS late_orders,
    ROUND(SUM(CASE
            WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
            THEN 1 ELSE 0
        END) * 100.0 / COUNT(*), 2) AS late_pct,
    ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date)), 2) AS avg_delay_days
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY late_pct DESC
LIMIT 10;
"""

result3 = pd.read_sql_query(q3, conn)
result3

,customer_state,total_orders,late_orders,late_pct,avg_delay_days
0,AL,397,95,23.93,-8.03
1,MA,717,141,19.67,-8.89
2,PI,476,76,15.97,-10.63
3,CE,1279,196,15.32,-10.11
4,SE,335,51,15.22,-9.33
5,BA,3256,457,14.04,-10.10
6,RJ,12353,1664,13.47,-11.06
7,TO,274,35,12.77,-11.44
8,PA,946,117,12.37,-13.39
9,ES,1995,244,12.23,-9.80


# **Problem 4: Category-wise Delay Risk**

In [11]:
q4 = """
SELECT
    ct.product_category_name_english AS category,
    COUNT(*) AS total_orders,
    ROUND(SUM(CASE
            WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
            THEN 1 ELSE 0
        END) * 100.0 / COUNT(*), 2) AS late_pct,
    ROUND(AVG(oi.price), 2) AS avg_price
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products pr ON oi.product_id = pr.product_id
JOIN category_translation ct ON pr.product_category_name = ct.product_category_name
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY category
HAVING total_orders >= 30
ORDER BY late_pct DESC
LIMIT 10;
"""

result4 = pd.read_sql_query(q4, conn)
result4

,category,total_orders,late_pct,avg_price
0,home_comfort_2,30,16.67,25.34
1,furniture_mattress_and_upholstery,37,13.51,116.85
2,audio,362,12.71,139.70
3,fashion_underwear_beach,127,12.60,73.28
4,christmas_supplies,150,12.00,58.25
5,books_technical,263,11.03,71.11
6,home_confort,429,10.26,135.22
7,construction_tools_lights,301,9.97,132.75
8,food,499,9.82,57.58
9,electronics,2729,9.75,56.81


In [12]:
q5 = """
SELECT
    payment_type,
    COUNT(*) AS total_transactions,
    ROUND(AVG(payment_value), 2) AS avg_payment_value
FROM order_payments
GROUP BY payment_type
ORDER BY avg_payment_value DESC;
"""

result5 = pd.read_sql_query(q5, conn)
result5

,payment_type,total_transactions,avg_payment_value
0,credit_card,76795,163.32
1,boleto,19784,145.03
2,debit_card,1529,142.57
3,voucher,5775,65.70
4,not_defined,3,0.00


In [13]:
import shutil
shutil.copy('olist.db', '/content/drive/MyDrive/SQL PROJECT /olist.db')
print("DB saved to Drive ✅")

DB saved to Drive ✅


In [17]:
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px

# Page config
st.set_page_config(page_title="Olist Delivery Performance Dashboard", layout="wide")

# Database connect
conn = sqlite3.connect('olist.db', check_same_thread=False)

st.title("📦 Olist E-Commerce: Delivery Performance & Customer Satisfaction")
st.markdown("Analyzing how delivery delays impact customer satisfaction, seller performance, and regional risk.")

# KPI Row
col1, col2, col3, col4 = st.columns(4)

kpi_query = """
SELECT
    COUNT(*) AS total_orders,
    ROUND(AVG(review_score), 2) AS avg_review
FROM orders o
JOIN order_review r ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL;
"""
kpi = pd.read_sql_query(kpi_query, conn)

col1.metric("Total Delivered Orders", f"{kpi['total_orders'][0]:,}")
col2.metric("Avg Review Score", kpi['avg_review'][0])

# Tabs
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Delay vs Satisfaction", "Seller Performance",
    "Regional Risk", "Category Risk", "Payment Methods"
])

with tab1:
    st.subheader("Delivery Delay Impact on Review Score")
    q1 = """
    SELECT
    CASE
        WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <= 0 THEN 'On-Time'
        WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) BETWEEN 1 AND 3 THEN '1-3 Days Late'
        WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) BETWEEN 4 AND 7 THEN '4-7 Days Late'
        ELSE '7+ Days Late'
    END AS delay_bucket,
    COUNT(*) AS total_orders,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM orders o
JOIN order_review r ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY delay_bucket
ORDER BY avg_review_score DESC;
     """
    df1 = pd.read_sql_query(q1, conn)
    fig1 = px.bar(df1, x='delay_bucket', y='avg_review_score',
                  color='avg_review_score', color_continuous_scale='RdYlGn')
    st.plotly_chart(fig1, use_container_width=True)

with tab2:
    st.subheader("Top & Worst Performing Sellers")
    q2 = """
    WITH seller_stats AS (
    SELECT
        oi.seller_id,
        COUNT(DISTINCT o.order_id) AS total_orders,
        SUM(CASE
                WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
                THEN 1 ELSE 0
            END) AS late_orders,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    JOIN order_review r ON o.order_id = r.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
    GROUP BY oi.seller_id
    HAVING total_orders >= 10
)
SELECT *,
       ROUND(1.0 - (late_orders * 1.0 / total_orders), 3) AS on_time_rate,
       RANK() OVER (ORDER BY (1.0 - (late_orders * 1.0 / total_orders)) DESC) AS seller_rank
FROM seller_stats
ORDER BY seller_rank
LIMIT 15;
     """
    df2 = pd.read_sql_query(q2, conn)
    st.dataframe(df2)

with tab3:
    st.subheader("State-wise Delivery Risk")
    q3 = """
    SELECT
    c.customer_state,
    COUNT(*) AS total_orders,
    SUM(CASE
            WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
            THEN 1 ELSE 0
        END) AS late_orders,
    ROUND(SUM(CASE
            WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
            THEN 1 ELSE 0
        END) * 100.0 / COUNT(*), 2) AS late_pct,
    ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date)), 2) AS avg_delay_days
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY late_pct DESC
LIMIT 10;
     """
    df3 = pd.read_sql_query(q3, conn)
    fig3 = px.bar(df3, x='customer_state', y='late_pct')
    st.plotly_chart(fig3, use_container_width=True)

with tab4:
    st.subheader("Category-wise Delay Risk")
    q4 = """
SELECT
    ct.product_category_name_english AS category,
    COUNT(*) AS total_orders,
    ROUND(SUM(CASE
            WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
            THEN 1 ELSE 0
        END) * 100.0 / COUNT(*), 2) AS late_pct,
    ROUND(AVG(oi.price), 2) AS avg_price
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products pr ON oi.product_id = pr.product_id
JOIN category_translation ct ON pr.product_category_name = ct.product_category_name
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY category
HAVING total_orders >= 30
ORDER BY late_pct DESC
LIMIT 10;
"""
    df4 = pd.read_sql_query(q4, conn)
    fig4 = px.bar(df4, x='category', y='late_pct')
    st.plotly_chart(fig4, use_container_width=True)

with tab5:
    st.subheader("Payment Method Distribution")
    q5 = """
SELECT
    payment_type,
    COUNT(*) AS total_transactions,
    ROUND(AVG(payment_value), 2) AS avg_payment_value
FROM order_payments
GROUP BY payment_type
ORDER BY avg_payment_value DESC;
"""
    df5 = pd.read_sql_query(q5, conn)
    fig5 = px.pie(df5, names='payment_type', values='total_transactions')
    st.plotly_chart(fig5, use_container_width=True)

2026-07-06 05:54:33.216 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-06 05:54:33.218 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-06 05:54:33.433 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-07-06 05:54:33.434 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-06 05:54:33.435 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-06 05:54:33.436 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-06 05:54:33.437 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [16]:
!pip install streamlit -q
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 48.8 MB/s eta 0:00:00
done


In [18]:
from google.colab import files
files.download('olist.db')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
!git config --global user.email "siddharthsingh262005@gmail.com"
!git config --global user.name "siddharth-0309"


!git clone https://github.com/siddharth-0309/olist-delivery-dashboard.git

Cloning into 'olist-delivery-dashboard'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 12 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 5.73 KiB | 5.73 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [24]:
import shutil
shutil.copy('olist.db', 'olist-delivery-dashboard/olist.db')
shutil.copy('app.py', 'olist-delivery-dashboard/app.py')

with open('olist-delivery-dashboard/requirements.txt', 'w') as f:
    f.write("streamlit\npandas\nplotly\n")

In [23]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px

st.set_page_config(page_title="Olist Delivery Performance Dashboard", layout="wide")

conn = sqlite3.connect('olist.db', check_same_thread=False)

st.title("📦 Olist E-Commerce: Delivery Performance & Customer Satisfaction")
st.markdown("Analyzing how delivery delays impact customer satisfaction, seller performance, and regional risk.")

col1, col2, col3, col4 = st.columns(4)

kpi_query = """
SELECT
    COUNT(*) AS total_orders,
    ROUND(AVG(review_score), 2) AS avg_review
FROM orders o
JOIN order_review r ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL;
"""
kpi = pd.read_sql_query(kpi_query, conn)

col1.metric("Total Delivered Orders", f"{kpi['total_orders'][0]:,}")
col2.metric("Avg Review Score", kpi['avg_review'][0])

tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Delay vs Satisfaction", "Seller Performance",
    "Regional Risk", "Category Risk", "Payment Methods"
])

with tab1:
    st.subheader("Delivery Delay Impact on Review Score")
    q1 = """
    SELECT
        CASE
            WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <= 0 THEN 'On-Time'
            WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) BETWEEN 1 AND 3 THEN '1-3 Days Late'
            WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) BETWEEN 4 AND 7 THEN '4-7 Days Late'
            ELSE '7+ Days Late'
        END AS delay_bucket,
        COUNT(*) AS total_orders,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM orders o
    JOIN order_review r ON o.order_id = r.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
    GROUP BY delay_bucket
    ORDER BY avg_review_score DESC;
    """
    df1 = pd.read_sql_query(q1, conn)
    fig1 = px.bar(df1, x='delay_bucket', y='avg_review_score',
                  color='avg_review_score', color_continuous_scale='RdYlGn')
    st.plotly_chart(fig1, width='stretch')

with tab2:
    st.subheader("Top & Worst Performing Sellers")
    q2 = """
    WITH seller_stats AS (
        SELECT
            oi.seller_id,
            COUNT(DISTINCT o.order_id) AS total_orders,
            SUM(CASE
                    WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
                    THEN 1 ELSE 0
                END) AS late_orders,
            ROUND(AVG(r.review_score), 2) AS avg_review_score
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN order_review r ON o.order_id = r.order_id
        WHERE o.order_delivered_customer_date IS NOT NULL
        GROUP BY oi.seller_id
        HAVING total_orders >= 10
    )
    SELECT *,
           ROUND(1.0 - (late_orders * 1.0 / total_orders), 3) AS on_time_rate,
           RANK() OVER (ORDER BY (1.0 - (late_orders * 1.0 / total_orders)) DESC) AS seller_rank
    FROM seller_stats
    ORDER BY seller_rank
    LIMIT 15;
    """
    df2 = pd.read_sql_query(q2, conn)
    st.dataframe(df2)

with tab3:
    st.subheader("State-wise Delivery Risk")
    q3 = """
    SELECT
        c.customer_state,
        COUNT(*) AS total_orders,
        SUM(CASE
                WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
                THEN 1 ELSE 0
            END) AS late_orders,
        ROUND(SUM(CASE
                WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
                THEN 1 ELSE 0
            END) * 100.0 / COUNT(*), 2) AS late_pct,
        ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date)), 2) AS avg_delay_days
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_delivered_customer_date IS NOT NULL
    GROUP BY c.customer_state
    ORDER BY late_pct DESC
    LIMIT 10;
    """
    df3 = pd.read_sql_query(q3, conn)
    fig3 = px.bar(df3, x='customer_state', y='late_pct')
    st.plotly_chart(fig3, width='stretch')

with tab4:
    st.subheader("Category-wise Delay Risk")
    q4 = """
    SELECT
        ct.product_category_name_english AS category,
        COUNT(*) AS total_orders,
        ROUND(SUM(CASE
                WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
                THEN 1 ELSE 0
            END) * 100.0 / COUNT(*), 2) AS late_pct,
        ROUND(AVG(oi.price), 2) AS avg_price
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    JOIN products pr ON oi.product_id = pr.product_id
    JOIN category_translation ct ON pr.product_category_name = ct.product_category_name
    WHERE o.order_delivered_customer_date IS NOT NULL
    GROUP BY category
    HAVING total_orders >= 30
    ORDER BY late_pct DESC
    LIMIT 10;
    """
    df4 = pd.read_sql_query(q4, conn)
    fig4 = px.bar(df4, x='category', y='late_pct')
    st.plotly_chart(fig4, width='stretch')

with tab5:
    st.subheader("Payment Method Distribution")
    q5 = """
    SELECT
        payment_type,
        COUNT(*) AS total_transactions,
        ROUND(AVG(payment_value), 2) AS avg_payment_value
    FROM order_payments
    GROUP BY payment_type
    ORDER BY avg_payment_value DESC;
    """
    df5 = pd.read_sql_query(q5, conn)
    fig5 = px.pie(df5, names='payment_type', values='total_transactions')
    st.plotly_chart(fig5, width='stretch')

Overwriting app.py


In [25]:
!git config --global user.email "siddharthsingh262005@gmail.com"
!git config --global user.name "siddharth-0309"

In [31]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

%cd olist-delivery-dashboard
!git add .
!git commit -m "Add app.py, olist.db, requirements.txt"
!git push https://{token}@github.com/siddharth-0309/olist-delivery-dashboard.git

[Errno 2] No such file or directory: 'olist-delivery-dashboard'
/content/olist-delivery-dashboard
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 30.84 MiB | 5.03 MiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
remote: warning: See https://gh.io/lfs for more information.
remote: warning: File olist.db is 65.17 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.
To https://github.com/siddharth-0309/olist-delivery-dashboard.git
   293eaf7..9c9c142  main -> main
